In [0]:
%pip install openpyxl --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
import pandas as pd
import io
import re

# Configuration
S3_BUCKET_PATH = "s3://ine-data/"
TARGET_DATABASE = "covid19.bronze"

excel_files = [
    "defunciones-2018.xlsx",
    "defunciones-2019.xlsx",
    "defunciones-2020.xlsx",
    "defunciones-2021.xlsx",
    "defunciones-2022.xlsx",
    "defunciones-2023.xlsx",
    "defunciones-2024.xlsx"
]

print(f"Starting Native-backed Bronze Pipeline pointing to: {TARGET_DATABASE}...")

for file_name in excel_files:
    full_path = f"{S3_BUCKET_PATH}{file_name}"
    year = re.search(r'\d{4}', file_name).group()
    target_table_name = f"{TARGET_DATABASE}.ine_defunciones_{year}"
    
    print(f"\n--- Processing file: {file_name} -> Destination: {target_table_name} ---")
    
    try:
        binary_df = spark.read.format("binaryFile").load(full_path)
        raw_binary_content = binary_df.select("content").collect()[0][0]
        pandas_df = pd.read_excel(io.BytesIO(raw_binary_content), engine="openpyxl")
        pandas_df.columns = [re.sub(r'[ ,;{}()\n\t=]', '_', str(col)) for col in pandas_df.columns]
        spark_df = spark.createDataFrame(pandas_df)
        spark_df = (spark_df
                    .withColumn("bronze_processing_date", F.current_timestamp())
                    .withColumn("source_file", F.lit(file_name)))

        print(f"Writing Delta table...")
        (spark_df.write
         .format("delta")
         .mode("overwrite") 
         .saveAsTable(target_table_name))
        
        print(f"Table {target_table_name} created successfully!")
            
    except Exception as e:
        print(f"Error processing file {file_name}: {str(e)}")

print("\nPipeline finished successfully!")